In [2]:
import zipfile
import pandas as pd
import os

ZIP_PATH = r"E:\Network_Intrusion\ml_models\__pycache__\Network dataset.zip"
EXTRACT_PATH = "dataset"

# Extract ZIP
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("ZIP extracted successfully")


ZIP extracted successfully


In [3]:
csv_files = []

for root, dirs, files in os.walk(EXTRACT_PATH):
    for file in files:
        if file.endswith(".csv"):
            csv_files.append(os.path.join(root, file))

print("CSV files found:", len(csv_files))

df_list = [pd.read_csv(f) for f in csv_files]
df = pd.concat(df_list, ignore_index=True)

print("Final dataset shape:", df.shape)


CSV files found: 8
Final dataset shape: (2830743, 79)


In [7]:
import zipfile, os
import pandas as pd

ZIP_PATH = r"E:\Network_Intrusion\ml_models\__pycache__\Network dataset.zip"
EXTRACT_PATH = "cicids"

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_PATH)

csv_files = []
for root, _, files in os.walk(EXTRACT_PATH):
    for f in files:
        if f.endswith(".csv"):
            csv_files.append(os.path.join(root, f))

print("CSV files:", len(csv_files))

df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
print("Dataset shape:", df.shape)


CSV files: 8
Dataset shape: (2830743, 79)


In [8]:
df.columns = df.columns.str.strip()
print("Label column exists:", "Label" in df.columns)


Label column exists: True


In [9]:
df = df.drop(columns=["Destination Port"], errors="ignore")


In [10]:
import numpy as np

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

print("After cleaning:", df.shape)


After cleaning: (2827876, 78)


In [11]:
X = df.drop(columns=["Label"])
y = df["Label"]

print("Classes:", y.unique())
print("Total classes:", y.nunique())


Classes: ['BENIGN' 'DDoS' 'PortScan' 'Bot' 'Infiltration'
 'Web Attack � Brute Force' 'Web Attack � XSS'
 'Web Attack � Sql Injection' 'FTP-Patator' 'SSH-Patator' 'DoS slowloris'
 'DoS Slowhttptest' 'DoS Hulk' 'DoS GoldenEye' 'Heartbleed']
Total classes: 15


In [12]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

for i, cls in enumerate(label_encoder.classes_):
    print(i, "→", cls)


0 → BENIGN
1 → Bot
2 → DDoS
3 → DoS GoldenEye
4 → DoS Hulk
5 → DoS Slowhttptest
6 → DoS slowloris
7 → FTP-Patator
8 → Heartbleed
9 → Infiltration
10 → PortScan
11 → SSH-Patator
12 → Web Attack � Brute Force
13 → Web Attack � Sql Injection
14 → Web Attack � XSS


In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)


In [17]:
X = X.select_dtypes(include=["int64", "float64"])


MemoryError: Unable to allocate 1.12 GiB for an array with shape (53, 2827876) and data type int64

In [18]:
X = X.astype("float32")


In [19]:
from sklearn.model_selection import train_test_split

X, _, y_encoded, _ = train_test_split(
    X,
    y_encoded,
    train_size=300_000,   # do NOT exceed this
    stratify=y_encoded,
    random_state=42
)


MemoryError: Unable to allocate 511. MiB for an array with shape (53, 2527876) and data type float32

In [20]:
X = X.select_dtypes(include=["int64", "float64"])
X = X.astype("float32")


In [21]:
import pandas as pd

data = X.copy()
data["Label"] = y_encoded


In [22]:
SAMPLE_SIZE = 200_000   # safe on 8 GB RAM

data_sampled = (
    data
    .groupby("Label", group_keys=False)
    .apply(lambda x: x.sample(
        n=min(len(x), SAMPLE_SIZE // data["Label"].nunique()),
        random_state=42
    ))
)

print("Sampled shape:", data_sampled.shape)


Sampled shape: (92935, 1)


C:\Users\sofiy\AppData\Local\Temp\ipykernel_33688\757828366.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data


In [23]:
X_sampled = data_sampled.drop(columns=["Label"])
y_sampled = data_sampled["Label"]


In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_sampled,
    y_sampled,
    test_size=0.2,
    stratify=y_sampled,
    random_state=42
)


In [25]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)


ValueError: at least one array or dtype is required

In [26]:
print("X_train type:", type(X_train))
print("X_train shape:", X_train.shape)
print("X_train columns:", X_train.columns if hasattr(X_train, "columns") else "No columns")
print("y_train length:", len(y_train))


X_train type: <class 'pandas.core.frame.DataFrame'>
X_train shape: (74348, 0)
X_train columns: Index([], dtype='object')
y_train length: 74348


In [28]:
import numpy as np

SAMPLE_SIZE = 200_000

data = pd.concat(
    [X.reset_index(drop=True),
     pd.Series(y_encoded, name="Label")],
    axis=1
)

num_classes = data["Label"].nunique()
per_class_target = SAMPLE_SIZE // num_classes

data_sampled = (
    data
    .groupby("Label", group_keys=False)
    .apply(lambda x: x.sample(
        n=min(len(x), per_class_target),  # 🔑 KEY FIX
        random_state=42
    ))
)

print("Sampled shape:", data_sampled.shape)
print("Class distribution:")
print(data_sampled["Label"].value_counts())


Sampled shape: (92935, 1)
Class distribution:
Label
0     13333
2     13333
4     13333
10    13333
3     10293
7      7935
11     5897
6      5796
5      5499
1      1956
12     1507
14      652
9        36
13       21
8        11
Name: count, dtype: int64


C:\Users\sofiy\AppData\Local\Temp\ipykernel_33688\1292679447.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data


In [29]:
X_sampled = data_sampled.drop(columns=["Label"])
y_sampled = data_sampled["Label"]


In [30]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_sampled,
    y_sampled,
    test_size=0.2,
    stratify=y_sampled,
    random_state=42
)


In [32]:
import numpy as np

X = X.select_dtypes(include=[np.number])


In [33]:
X = X.astype("float32")


In [34]:
print("Number of features after selection:", X.shape[1])
print("Sample columns:", X.columns[:10])


Number of features after selection: 0
Sample columns: Index([], dtype='object')


In [35]:
# ALWAYS start from the original dataframe
X = df.drop(columns=["Label"])
y = df["Label"]


In [36]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)


In [37]:
import numpy as np

X = X.select_dtypes(include=[np.number])


In [38]:
print("Number of features after selection:", X.shape[1])
print("First 10 columns:", X.columns[:10])


Number of features after selection: 77
First 10 columns: Index(['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
       'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
       'Fwd Packet Length Max', 'Fwd Packet Length Min',
       'Fwd Packet Length Mean', 'Fwd Packet Length Std',
       'Bwd Packet Length Max'],
      dtype='object')


In [39]:
X = X.astype("float32")


In [40]:
data = pd.concat(
    [X.reset_index(drop=True),
     pd.Series(y_encoded, name="Label")],
    axis=1
)


In [41]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Total classes:", len(label_encoder.classes_))
print(label_encoder.classes_)


Total classes: 15
['BENIGN' 'Bot' 'DDoS' 'DoS GoldenEye' 'DoS Hulk' 'DoS Slowhttptest'
 'DoS slowloris' 'FTP-Patator' 'Heartbleed' 'Infiltration' 'PortScan'
 'SSH-Patator' 'Web Attack � Brute Force' 'Web Attack � Sql Injection'
 'Web Attack � XSS']


In [42]:
import pandas as pd

data = pd.concat(
    [X.reset_index(drop=True),
     pd.Series(y_encoded, name="Label")],
    axis=1
)

SAMPLE_SIZE = 200_000   # safe on laptop
num_classes = data["Label"].nunique()
per_class = SAMPLE_SIZE // num_classes

data_sampled = (
    data
    .groupby("Label", group_keys=False)
    .apply(lambda x: x.sample(
        n=min(len(x), per_class),
        random_state=42
    ))
)

print("Sampled shape:", data_sampled.shape)
print(data_sampled["Label"].value_counts())


Sampled shape: (92935, 78)
Label
0     13333
2     13333
4     13333
10    13333
3     10293
7      7935
11     5897
6      5796
5      5499
1      1956
12     1507
14      652
9        36
13       21
8        11
Name: count, dtype: int64


C:\Users\sofiy\AppData\Local\Temp\ipykernel_33688\1928125266.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data


In [43]:
X_sampled = data_sampled.drop(columns=["Label"])
y_sampled = data_sampled["Label"]


In [44]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_sampled,
    y_sampled,
    test_size=0.2,
    stratify=y_sampled,
    random_state=42
)

print("X_train shape:", X_train.shape)


X_train shape: (74348, 77)


In [45]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)


RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42)

In [46]:
from sklearn.metrics import classification_report, accuracy_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))


Accuracy: 0.9891859902082101
                            precision    recall  f1-score   support

                    BENIGN       0.99      0.99      0.99      2667
                       Bot       0.98      0.98      0.98       391
                      DDoS       1.00      1.00      1.00      2667
             DoS GoldenEye       1.00      1.00      1.00      2059
                  DoS Hulk       1.00      1.00      1.00      2667
          DoS Slowhttptest       1.00      0.99      0.99      1100
             DoS slowloris       0.99      1.00      1.00      1159
               FTP-Patator       1.00      1.00      1.00      1587
                Heartbleed       1.00      1.00      1.00         2
              Infiltration       1.00      1.00      1.00         7
                  PortScan       1.00      1.00      1.00      2667
               SSH-Patator       1.00      1.00      1.00      1179
  Web Attack � Brute Force       0.73      0.82      0.77       301
Web Attack � Sql I

In [47]:
import pickle

pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(label_encoder, open("label_encoder.pkl", "wb"))

print("✅ Model saved successfully")


✅ Model saved successfully


In [48]:
import pandas as pd

feature_importance = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(feature_importance.head(15))


Init_Win_bytes_backward        0.066742
Init_Win_bytes_forward         0.033169
min_seg_size_forward           0.031652
Bwd Packets/s                  0.029406
Flow IAT Mean                  0.027722
Flow Packets/s                 0.027482
Subflow Bwd Bytes              0.024177
Flow IAT Max                   0.023553
Fwd Packet Length Max          0.023525
Flow Duration                  0.023100
Bwd Packet Length Max          0.022331
Total Length of Fwd Packets    0.022317
Flow IAT Std                   0.022275
Max Packet Length              0.022180
Packet Length Mean             0.022094
dtype: float64


In [49]:
SELECTED_FEATURES = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Fwd Packet Length Mean",
    "Bwd Packet Length Mean",
    "Packet Length Variance"
]


In [50]:
X_small = df[SELECTED_FEATURES].astype("float32")

y = df["Label"]
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Sample again (safe)
data = pd.concat(
    [X_small.reset_index(drop=True),
     pd.Series(y_encoded, name="Label")],
    axis=1
)

SAMPLE_SIZE = 150_000
per_class = SAMPLE_SIZE // data["Label"].nunique()

data_sampled = (
    data
    .groupby("Label", group_keys=False)
    .apply(lambda x: x.sample(
        n=min(len(x), per_class),
        random_state=42
    ))
)

X_final = data_sampled.drop(columns=["Label"])
y_final = data_sampled["Label"]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final,
    test_size=0.2,
    stratify=y_final,
    random_state=42
)

from sklearn.ensemble import RandomForestClassifier
deploy_model = RandomForestClassifier(
    n_estimators=150,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

deploy_model.fit(X_train, y_train)


C:\Users\sofiy\AppData\Local\Temp\ipykernel_33688\346974385.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data


RandomForestClassifier(class_weight='balanced', n_estimators=150, n_jobs=-1,
                       random_state=42)

In [51]:
import pickle

pickle.dump(deploy_model, open("deploy_model.pkl", "wb"))
pickle.dump(label_encoder, open("label_encoder.pkl", "wb"))

print("✅ Deployment model saved")


✅ Deployment model saved


In [77]:
sample = X_test.iloc[[0]]
pred = deploy_model.predict(sample)[0]

print("Prediction:", label_encoder.inverse_transform([pred])[0])


Prediction: BENIGN


In [55]:
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 6 deployment features (EXACT names)
DEPLOY_FEATURES = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Fwd Packet Length Mean",
    "Bwd Packet Length Mean",
    "Packet Length Variance",
]

# extract features
X_small = df[DEPLOY_FEATURES].astype("float32")

# 🔥 APPLY LOG TRANSFORM DURING TRAINING
X_small = np.log1p(X_small)

y = df["Label"]
le = LabelEncoder()
y_enc = le.fit_transform(y)

# balanced sampling
data = X_small.copy()
data["Label"] = y_enc

SAMPLE_SIZE = 150_000
per_class = SAMPLE_SIZE // data["Label"].nunique()

data = (
    data.groupby("Label", group_keys=False)
        .apply(lambda x: x.sample(
            n=min(len(x), per_class),
            random_state=42
        ))
)

X_final = data.drop(columns=["Label"])
y_final = data["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final,
    test_size=0.2,
    stratify=y_final,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# 🔐 SAVE EVERYTHING
model.feature_names_ = DEPLOY_FEATURES
model.log_transform = True

pickle.dump(model, open("deploy_model.pkl", "wb"))
pickle.dump(le, open("label_encoder.pkl", "wb"))

print("✅ Deployment model retrained correctly")


e:\Network_Intrusion\asmi\lib\site-packages\pandas\core\internals\blocks.py:395: RuntimeWarning: divide by zero encountered in log1p
  result = func(self.values, **kwargs)
e:\Network_Intrusion\asmi\lib\site-packages\pandas\core\internals\blocks.py:395: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)
C:\Users\sofiy\AppData\Local\Temp\ipykernel_33688\1695686174.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data.groupby("Label", group_keys=False)


ValueError: Input X contains infinity or a value too large for dtype('float32').

In [56]:
import numpy as np

# replace invalid negatives used as placeholders
X_small = X_small.replace([-1, -np.inf, np.inf], 0)

# ensure no negatives
X_small[X_small < 0] = 0


In [57]:
X_small = np.log1p(X_small)


In [58]:
assert np.isfinite(X_small.values).all(), "NaN or inf found after log transform"
print("✅ Log transform clean")


AssertionError: NaN or inf found after log transform

In [59]:
import numpy as np

X_small = df[DEPLOY_FEATURES].copy()

# force everything to numeric
X_small = X_small.apply(pd.to_numeric, errors="coerce")


In [60]:
X_small = X_small.replace([np.inf, -np.inf], np.nan)
X_small = X_small.fillna(0)

# ensure no negatives
X_small[X_small < 0] = 0


In [61]:
X_small = X_small.clip(upper=1e7)


In [62]:
X_small = X_small.astype("float64")


In [63]:
X_small = np.log1p(X_small)


In [64]:
X_small = X_small.replace([np.inf, -np.inf], 0)
X_small = X_small.fillna(0)


In [65]:
assert np.isfinite(X_small.values).all()
print("✅ Log transform clean")


✅ Log transform clean


In [66]:
data = X_small.copy()
data["Label"] = y_enc
# sampling → split → train → save


In [68]:
X_small = np.log1p(X_small)


In [69]:
X_small = X_small.astype("float64")


In [70]:
import numpy as np
assert np.isfinite(X_small.values).all()
print("✅ Log transform clean & float64")


✅ Log transform clean & float64


In [71]:
# ---- feature extraction ----
X_small = df[DEPLOY_FEATURES].copy()

# force numeric
X_small = X_small.apply(pd.to_numeric, errors="coerce")

# clean invalid values
import numpy as np
X_small = X_small.replace([np.inf, -np.inf], np.nan)
X_small = X_small.fillna(0)
X_small[X_small < 0] = 0

# clip extremes
X_small = X_small.clip(upper=1e7)

# log transform
X_small = np.log1p(X_small)

# 🔥 KEEP FLOAT64
X_small = X_small.astype("float64")

# final safety check
assert np.isfinite(X_small.values).all()
print("✅ Log transform clean & float64")

# ---- labels ----
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_enc = le.fit_transform(df["Label"])

# ---- sampling ----
data = X_small.copy()
data["Label"] = y_enc

SAMPLE_SIZE = 150_000
per_class = SAMPLE_SIZE // data["Label"].nunique()

data = (
    data.groupby("Label", as_index=False)
        .apply(lambda x: x.sample(
            n=min(len(x), per_class),
            random_state=42
        ))
        .reset_index(drop=True)
)

X_final = data.drop(columns=["Label"])
y_final = data["Label"]

# ---- split ----
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final,
    test_size=0.2,
    stratify=y_final,
    random_state=42
)

# ---- train ----
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)   # ✅ THIS WILL NOW WORK


✅ Log transform clean & float64


C:\Users\sofiy\AppData\Local\Temp\ipykernel_33688\663513097.py:39: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data.groupby("Label", as_index=False)


RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42)

In [72]:
model.feature_names_ = DEPLOY_FEATURES
model.log_transform = True

import pickle
pickle.dump(model, open("deploy_model.pkl", "wb"))
pickle.dump(le, open("label_encoder.pkl", "wb"))

print("✅ Deployment model trained successfully")


✅ Deployment model trained successfully


In [73]:
sample = X_test.iloc[[0]]
pred = deploy_model.predict(sample)[0]

print("Prediction:", label_encoder.inverse_transform([pred])[0])


Prediction: BENIGN


In [74]:
sample = X_test.iloc[[0]]
pred = deploy_model.predict(sample)[0]

print("Prediction:", label_encoder.inverse_transform([pred])[0])


Prediction: BENIGN


In [75]:
sample = X_test.iloc[[0]]
pred = deploy_model.predict(sample)[0]

print("Prediction:", label_encoder.inverse_transform([pred])[0])


Prediction: BENIGN


In [78]:
import pickle
import numpy as np

model = pickle.load(open("deploy_model.pkl", "rb"))
le = pickle.load(open("label_encoder.pkl", "rb"))

print("Classes:", le.classes_)
print("Features:", model.feature_names_)
print("Uses log:", model.log_transform)

# 🔥 HARD-CODED ATTACK SAMPLE (from CIC-IDS stats)
sample = np.array([[200, 4000, 15, 40, 30, 5]], dtype="float64")

if model.log_transform:
    sample = np.log1p(sample)

pred = model.predict(sample)[0]
proba = model.predict_proba(sample)[0]

print("Prediction:", le.inverse_transform([pred])[0])
print("Probabilities:")
for c, p in zip(le.classes_, proba):
    print(c, round(p, 4))


Classes: ['BENIGN' 'Bot' 'DDoS' 'DoS GoldenEye' 'DoS Hulk' 'DoS Slowhttptest'
 'DoS slowloris' 'FTP-Patator' 'Heartbleed' 'Infiltration' 'PortScan'
 'SSH-Patator' 'Web Attack � Brute Force' 'Web Attack � Sql Injection'
 'Web Attack � XSS']
Features: ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packet Length Mean', 'Bwd Packet Length Mean', 'Packet Length Variance']
Uses log: True


e:\Network_Intrusion\asmi\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Prediction: BENIGN
Probabilities:
BENIGN 0.65
Bot 0.025
DDoS 0.005
DoS GoldenEye 0.0
DoS Hulk 0.015
DoS Slowhttptest 0.0
DoS slowloris 0.005
FTP-Patator 0.05
Heartbleed 0.04
Infiltration 0.2
PortScan 0.0
SSH-Patator 0.005
Web Attack � Brute Force 0.005
Web Attack � Sql Injection 0.0
Web Attack � XSS 0.0


e:\Network_Intrusion\asmi\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
